In [ ]:
############### Exercise 1: Bitcoin Agent ###############
import requests

def get_bitcoin_price():
    url = "https://api.binance.com/api/v3/ticker/price?symbol=BTCUSDT"
    response = requests.get(url)
    data = response.json()
    return float(data["price"])

# Simulated chat
user_input = input("User: ")
if "price" in user_input.lower():
    price = get_bitcoin_price()
    print(f"AI: The current price of Bitcoin (BTC) is ${price:,.2f} USD. Prices can fluctuate rapidly, so check a reliable source for the most up-to-date information.")
else:
    print("AI: I can only tell you the Bitcoin price right now.")


In [ ]:
################ Exercise 1: Bitcoin Agent using LLM #################

from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langchain.tools import tool
import requests

BINANCE_API = "https://api.binance.com/api/v3/ticker/price?symbol=BTCUSDT"


@tool
def check_btc_price() -> None:
    """
    To check the price of BTC now

    Returns:
        price (str): The price of BTC now
    """
    response = requests.get(BINANCE_API)
    result = response.json()
    return result["price"]


tools = [check_btc_price]

llm = ChatOllama(
    model="qwen3",
    temperature=0,
)

sys_prompt = f"""You are a helpful and accurate Qwen3 AI assistant deployed locally via Ollama.
You are provided a tool to lookup the latest price of bitcoin (BTC).
Use it only when necessary."""

agent = create_agent(model=llm, tools=tools, system_prompt=sys_prompt)

query = "What is the price of bitcoin now?"

response = agent.invoke({"messages": [{"role": "user", "content": query}]})
response["messages"][-1].content

In [ ]:
############### Exercise 2: Bitcoin Agent Using MCP ###############

import asyncio
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient   # note the underscore version

async def run_agent():
    client = MultiServerMCPClient(
        {
            "weather": {
                "transport": "streamable_http",
                "url": "http://127.0.0.1:40000/mcp",

            }
        }
    )

    tools = await client.get_tools()

    llm = ChatOllama(model="qwen3", temperature=0)

    sys_prompt = """You are a helpful and accurate Qwen3 AI assistant deployed locally via Ollama.
    You are provided a tool to lookup the latest price of bitcoin (BTC).
    Use it only when necessary."""

    agent = create_agent(model=llm, tools=tools, system_prompt=sys_prompt)

    query = "What is the price of bitcoin now?"
    response = await agent.ainvoke({"messages": [{"role": "user", "content": query}]})
    print(response["messages"][-1].content)

# In a notebook, call the async function directly with await:
await run_agent()


In [ ]:
##################### Exercise 3: Workflow #####################

import json
import requests
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langchain.tools import tool


class GraphState(TypedDict):
    input_text: str
    related: bool
    output: str


####### Step 1 ######

check_llm = ChatOllama(model="qwen3", temperature=0, format="json")

check_sys_prompt = """You are a helpful and accurate Qwen3 AI assistant. 
Check if the user message is related to crypto or finance.

Output in the following JSON format
{
    "related": "Y or N"
}"""

check_agent = create_agent(model=check_llm, system_prompt=check_sys_prompt)


def detect_intention(state: GraphState) -> dict:
    user_msg = state["input_text"]
    response = check_agent.invoke({"messages": [{"role": "user", "content": user_msg}]})
    llm_output = response["messages"][-1].content
    result = json.loads(llm_output)
    return {"related": result["related"].strip().lower() == "y"}


####### End of Step 1 ######

####### Step 2a ######

chat_llm = ChatOllama(
    model="qwen3",
    temperature=0,
)

BINANCE_API = "https://api.binance.com/api/v3/ticker/price?symbol=BTCUSDT"


@tool
def check_btc_price() -> None:
    """
    To check the price of BTC now

    Returns:
        price (str): The price of BTC now
    """
    response = requests.get(BINANCE_API)
    result = response.json()
    return result["price"]


tools = [check_btc_price]

chat_sys_prompt = f"""You are a helpful and accurate Qwen3 AI assistant deployed locally via Ollama.
You are provided a tool to lookup the latest price of bitcoin (BTC).
Use it only when necessary."""

chat_agent = create_agent(model=chat_llm, tools=tools, system_prompt=chat_sys_prompt)


def generate_response(state: GraphState) -> dict:
    user_msg = state["input_text"]
    response = chat_agent.invoke({"messages": [{"role": "user", "content": user_msg}]})
    llm_output = response["messages"][-1].content
    return {"output": llm_output}


####### End of Step 2a ######

####### The workflow in graph ######

# Create the workflow graph
workflow = StateGraph(GraphState)

# Add Nodes
workflow.add_node("1", detect_intention)
workflow.add_node("2a", generate_response)
workflow.add_node("2b", lambda _: {"output": "Invalid topic."})

# Set Entry Point
workflow.set_entry_point("1")

# Add edges
workflow.add_conditional_edges("1", lambda x: x["related"], {True: "2a", False: "2b"})
workflow.add_edge("2a", END)
workflow.add_edge("2b", END)

app = workflow.compile()

####### End of the workflow ######

In [ ]:
result = app.invoke({"input_text": "What is the price of bitcoin now?"})
result

In [ ]:
result = app.invoke({"input_text": "Who are you?"})
result